# 18 — Multi-Tenant Chat Sessions: Actor Model vs. LangGraph

This notebook demonstrates how the actor runtime handles multiple concurrent users
with **full session isolation** — something that LangGraph/CrewAI cannot do structurally.

### What we'll show
| Section | What it demonstrates |
|---|---|
| 1 | Session isolation — `AgentId` key = session ID |
| 2 | Multi-turn memory — each session remembers its own history |
| 3 | Concurrent throughput — 500 sessions, one event loop, ~50ms mock LLM |
| 4 | Crash isolation — one session explodes, others keep going |
| 5 | Per-session streaming — each user gets their own token stream |
| 6 | Rate-limiting via backpressure — slow LLM, small mailbox, no floods |
| 7 | Session teardown — garbage collecting idle sessions |
| 8 | Why LangGraph fails for each of these |

> **Architecture note**: In the actor model `AgentId(type, key)` is both address and isolation boundary.
> `type='chat'` describes the behaviour; `key=session_id` makes it a unique actor.
> Two sessions share zero state — not even a Python object reference.

In [ ]:
import sys, asyncio, time, uuid, random
sys.path.insert(0, "..")

from raavan.logger import setup_logging
from raavan.core.runtime._local import LocalRuntime
from raavan.core.runtime._protocol import AgentId, TopicId
from raavan.core.runtime._types import MessageContext, RestartPolicy, StreamDone
from raavan.core.runtime._stream import StreamPublisher
from raavan.core.runtime._supervisor import Supervisor, SupervisorEscalation

setup_logging(mode="pretty")
print("ready")

## 1. Session Isolation — The Key Insight

The entire multi-tenancy story is one line:

```python
AgentId(type="chat", key=session_id)
```

- Same `type` → same handler code (registered once)
- Different `key` → completely different actor: separate mailbox, separate state, separate lifecycle

**LangGraph equivalent**: you'd need a separate `CompiledGraph` instance per user, stored in a dict you manage yourself, with manual thread IDs, checkpointers, and state serialization.

Here we register the handler **once** and route 5 sessions **automatically**.

In [ ]:
async def demo_isolation():
    rt = LocalRuntime()
    await rt.start()

    # Single handler registration — works for ALL sessions
    async def chat_handler(ctx: MessageContext, payload: str) -> str:
        return f"[{ctx.agent_id.key}] received: '{payload}'"

    await rt.register("chat", chat_handler)   # register ONCE

    print("=== Sending to 5 sessions concurrently ===")
    results = await asyncio.gather(
        rt.send_message("Hello!",    recipient=AgentId("chat", "alice")),
        rt.send_message("Bonjour!",  recipient=AgentId("chat", "bob")),
        rt.send_message("Hola!",     recipient=AgentId("chat", "carlos")),
        rt.send_message("Ciao!",     recipient=AgentId("chat", "diana")),
        rt.send_message("Konnichiwa!", recipient=AgentId("chat", "emi")),
    )

    for r in results:
        print(f"  {r}")

    print(f"\nActors spawned  : {len(rt._agents_started)}  (one per session, on demand)")
    print(f"Handler registered: 1  (shared code, isolated state)")
    print(f"Dispatcher agents : {[str(a) for a in rt._dispatcher.registered_agents]}")

    await rt.stop()

await demo_isolation()

## 2. Multi-Turn Memory — Per-Session History

Each actor carries its own session state across turns. The `ctx.agent_id.key` is the session ID.

Alice's turn 3 has no knowledge of Bob's turn 3. No locking, no session dict management — the actor *is* the session.

**LangGraph equivalent**: you must pass `config={"configurable": {"thread_id": session_id}}` to every `invoke()` call and maintain a checkpointer (SQLite or Postgres). Forgetting the thread_id → sessions silently share state.

In [ ]:
async def demo_multi_turn():
    rt = LocalRuntime()
    await rt.start()

    # Each session accumulates its own history dict entry
    histories: dict[str, list[str]] = {}

    async def chat_agent(ctx: MessageContext, payload: str) -> str:
        sid = ctx.agent_id.key
        if sid not in histories:
            histories[sid] = []
        histories[sid].append(payload)
        turn   = len(histories[sid])
        recent = histories[sid][-3:]           # last 3 messages as context
        return f"[{sid}|turn {turn}] context={recent} → mock response"

    await rt.register("chat", chat_agent)

    alice = AgentId("chat", "alice")
    bob   = AgentId("chat", "bob")

    # Interleaved turns — real concurrent chat
    turns = [
        (alice, "What is Python?"),
        (bob,   "Tell me about asyncio"),
        (alice, "How do I use decorators?"),
        (bob,   "What is an event loop?"),
        (alice, "Can you give an example?"),
        (bob,   "How does gather() work?"),
    ]

    print("=== Interleaved turns across 2 sessions ===")
    for agent, msg in turns:
        r = await rt.send_message(msg, recipient=agent)
        print(f"  {r}")

    print(f"\n=== Final histories (separate, never mixed) ===")
    print(f"  alice ({len(histories['alice'])} turns): {histories['alice']}")
    print(f"  bob   ({len(histories['bob'])} turns)  : {histories['bob']}")

    await rt.stop()

await demo_multi_turn()

## 3. Concurrent Throughput — 500 Sessions

This is where the actor model wins decisively over LangGraph.

With a **50ms mock LLM call** per session:
- **Sequential** (LangGraph default): 500 × 50ms = **25 seconds**
- **Actor model** (this): all 500 run concurrently = **~50ms + overhead**

Actors don't block each other — each waits in its own `await asyncio.sleep()`. The event loop interleaves them for free.

In [ ]:
async def demo_throughput():
    N = 500
    MOCK_LLM_MS = 50   # simulated LLM latency

    rt = LocalRuntime(mailbox_capacity=1000, send_timeout=60.0)
    await rt.start()

    response_times: list[float] = []

    async def llm_agent(ctx: MessageContext, payload: str) -> str:
        t0 = time.perf_counter()
        await asyncio.sleep(MOCK_LLM_MS / 1000)   # mock LLM call
        response_times.append((time.perf_counter() - t0) * 1000)
        return f"reply to: {payload}"

    await rt.register("chat", llm_agent)

    # Sequential baseline (10 sessions)
    t0 = time.perf_counter()
    for i in range(10):
        await rt.send_message(f"msg-{i}", recipient=AgentId("chat", f"seq-{i}"))
    seq_10 = (time.perf_counter() - t0) * 1000

    # Concurrent: full 500 sessions
    response_times.clear()
    t0 = time.perf_counter()
    results = await asyncio.gather(*[
        rt.send_message(f"user {i} says hello", recipient=AgentId("chat", f"user-{i}"))
        for i in range(N)
    ])
    conc_total = (time.perf_counter() - t0) * 1000

    avg_response = sum(response_times) / len(response_times)

    print(f"=== Throughput: {N} concurrent sessions ===")
    print(f"  Mock LLM latency (per call) : {MOCK_LLM_MS}ms")
    print(f"  Sequential estimate (10×)   : {seq_10:.0f}ms for 10 → ~{seq_10/10*N:.0f}ms for {N}")
    print(f"  Concurrent actual           : {conc_total:.0f}ms for all {N}")
    print(f"  Speedup                     : ~{seq_10/10*N/conc_total:.0f}x")
    print(f"  Avg per-handler latency     : {avg_response:.1f}ms  (close to {MOCK_LLM_MS}ms mock)")
    print(f"  Actors spawned              : {len(rt._agents_started)}")
    print(f"  All {N} replies received     : {len(results) == N and all('reply' in r for r in results)}")

    await rt.stop()

await demo_throughput()

## 4. Crash Isolation — One Session Explodes, Others Survive

In LangGraph, one crashing node raises inside `graph.invoke()` — you handle it with try/except, but **the whole execution graph is corrupted**. You must restart from scratch, losing all context.

With actors: each session is a separate actor. `HandlerError` propagates only to the caller of that session. All other sessions continue uninterrupted.

We'll run 10 sessions where session `crash-5` always raises a `RuntimeError`.

In [ ]:
async def demo_crash_isolation():
    rt = LocalRuntime()
    await rt.start()

    completed: list[str] = []
    crashed:   list[str] = []

    async def chat_agent(ctx: MessageContext, payload: str) -> str:
        sid = ctx.agent_id.key
        if sid == "crash-5":
            raise RuntimeError(f"OOM in session {sid}: GPU memory exhausted")
        await asyncio.sleep(0.01)
        return f"ok from {sid}"

    await rt.register("chat", chat_agent)

    # Run 10 sessions concurrently — session crash-5 will fail
    raw_results = await asyncio.gather(*[
        rt.send_message("hello", recipient=AgentId("chat", f"crash-{i}"))
        for i in range(10)
    ], return_exceptions=True)

    print("=== Results per session ===")
    for i, r in enumerate(raw_results):
        if isinstance(r, Exception):
            crashed.append(f"crash-{i}")
            print(f"  crash-{i}: FAILED  ← {type(r).__name__}: {r}")
        else:
            completed.append(f"crash-{i}")
            print(f"  crash-{i}: {r}")

    print(f"\n  Sessions completed: {len(completed)}/10")
    print(f"  Sessions failed   : {len(crashed)}/10  ← only the bad one")
    print(f"  Blast radius      : {len(crashed)/10*100:.0f}% of sessions affected")

    # Key point: send more messages to surviving sessions — they still work
    r = await rt.send_message("still alive?", recipient=AgentId("chat", "crash-0"))
    print(f"\n  crash-0 after crash-5 failed: '{r}'  ← unaffected")

    await rt.stop()

await demo_crash_isolation()

## 5. Per-Session Streaming — Each User Gets Their Own Token Stream

Real LLM responses are streamed token-by-token. With multiple users:

- **LangGraph**: `astream()` streams between nodes, not within. To stream tokens inside a node you need custom streaming callbacks, and there is no way to fan tokens to the right user without a dict mapping you manage.
- **Actor model**: Each session has a `TopicId("tokens", session_id)`. User subscribes to their own topic. Agent emits tokens — routing is automatic.

In [ ]:
async def demo_per_session_streaming():
    rt = LocalRuntime()
    await rt.start()

    # Each session has a dedicated token stream topic
    # Consumers register as subscribers to their own topic
    user_streams: dict[str, list] = {"alice": [], "bob": [], "carlos": []}

    async def alice_consumer(ctx: MessageContext, payload):
        user_streams["alice"].append(payload)

    async def bob_consumer(ctx: MessageContext, payload):
        user_streams["bob"].append(payload)

    async def carlos_consumer(ctx: MessageContext, payload):
        user_streams["carlos"].append(payload)

    await rt.register("consumer_alice",  alice_consumer)
    await rt.register("consumer_bob",    bob_consumer)
    await rt.register("consumer_carlos", carlos_consumer)

    # Each user subscribes only to their own session topic
    await rt.subscribe("consumer_alice",  TopicId("tokens", "alice"))
    await rt.subscribe("consumer_bob",    TopicId("tokens", "bob"))
    await rt.subscribe("consumer_carlos", TopicId("tokens", "carlos"))

    # LLM agent: streams tokens to the calling session's topic
    MOCK_RESPONSES = {
        "alice":  "Python is a high level language",
        "bob":    "Asyncio enables concurrent code",
        "carlos": "Actors isolate state naturally",
    }

    async def llm_streaming_agent(ctx: MessageContext, payload: str) -> str:
        sid   = ctx.agent_id.key
        topic = TopicId("tokens", sid)          # session-specific topic
        pub   = StreamPublisher(rt, topic, sender=ctx.agent_id)

        response = MOCK_RESPONSES.get(sid, "unknown session")
        for token in response.split():
            await pub.emit(token)
            await asyncio.sleep(0.01)           # simulate token delay
        await pub.close()                       # sends StreamDone
        return "stream_complete"

    await rt.register("chat", llm_streaming_agent)

    # All 3 sessions stream concurrently
    t0 = time.perf_counter()
    await asyncio.gather(
        rt.send_message("Explain Python",  recipient=AgentId("chat", "alice")),
        rt.send_message("Explain asyncio", recipient=AgentId("chat", "bob")),
        rt.send_message("Explain actors",  recipient=AgentId("chat", "carlos")),
    )
    await asyncio.sleep(0.05)   # drain remaining publish_message tasks
    elapsed = (time.perf_counter() - t0) * 1000

    print("=== Per-session token streams (no cross-contamination) ===")
    for user, tokens in user_streams.items():
        text_tokens = [t for t in tokens if not isinstance(t, StreamDone)]
        done        = any(isinstance(t, StreamDone) for t in tokens)
        print(f"  {user:8s}: tokens={text_tokens}  StreamDone={done}")

    print(f"\n  All 3 ran concurrently in {elapsed:.0f}ms")
    print(f"  Each user received ONLY their session's tokens — zero cross-talk")

    await rt.stop()

await demo_per_session_streaming()

## 6. Backpressure — Protecting a Slow LLM Endpoint

**LangGraph problem**: no built-in backpressure. If 1000 users hit at once and your LLM does 10 req/sec, LangGraph will happily fire all 1000 `invoke()` calls, saturate the API, and flood your process with `RateLimitError` exceptions that you catch one by one inside nodes.

**Actor model**: `Mailbox(capacity=N)` is the backpressure boundary. When the LLM actor's mailbox is full, new sends raise `MailboxFullError` immediately — the caller discovers the system is saturated before wasting resources.

In [ ]:
async def demo_backpressure():
    # Tiny mailbox: only 3 messages can queue at once behind the slow LLM actor
    rt = LocalRuntime(mailbox_capacity=3, send_timeout=5.0)
    await rt.start()

    queue_sizes: list[int] = []
    processed:   list[int] = []

    slow_id = AgentId("llm", "shared")

    async def slow_llm(ctx: MessageContext, payload) -> str:
        mb = rt._dispatcher.get_mailbox(slow_id)
        if mb:
            queue_sizes.append(mb.size)   # snapshot queue depth while handling
        await asyncio.sleep(0.08)         # 80ms mock LLM
        processed.append(payload)
        return f"done:{payload}"

    await rt.register("llm", slow_llm)

    # Fire 8 messages at a mailbox that can only hold 3
    # Some will succeed (queued), some will get MailboxFullError
    from raavan.core.runtime._mailbox import MailboxFullError

    tasks = [
        asyncio.create_task(rt.send_message(i, recipient=slow_id))
        for i in range(8)
    ]
    raw = await asyncio.gather(*tasks, return_exceptions=True)

    success = [r for r in raw if isinstance(r, str)]
    dropped = [r for r in raw if isinstance(r, Exception)]

    print("=== Backpressure with capacity=3 mailbox ===")
    print(f"  Sent           : 8 messages")
    print(f"  Processed      : {len(success)}  results={success}")
    print(f"  Dropped        : {len(dropped)}  (MailboxFullError — fast fail)")
    print(f"  Drop type      : {type(dropped[0]).__name__ if dropped else 'none'}")
    print(f"  Queue snapshots: {queue_sizes}  (never exceeded capacity)")
    print(f"  Max queue depth: {max(queue_sizes, default=0)}")
    print()
    print("  In LangGraph: all 8 would call the LLM node — you'd get 8 API calls")
    print("  and handle RateLimitError inside the node code, not at the boundary.")

    await rt.stop()

await demo_backpressure()

## 7. Session Teardown — Garbage Collecting Idle Sessions

In a real chat app, sessions come and go. With LangGraph you must track all `CompiledGraph` instances and delete them manually. Forget one and it leaks.

With actors: `rt.stop()` tears down everything. For fine-grained control, you can shut down a single session's actor by stopping the runtime and checking `_agents_started`. We demonstrate 500 sessions being created, used once, and the full set cleaned up.

In [ ]:
async def demo_session_lifecycle():
    rt = LocalRuntime(mailbox_capacity=1000, send_timeout=30.0)
    await rt.start()

    async def chat_agent(ctx: MessageContext, payload: str) -> str:
        await asyncio.sleep(0.005)
        return f"ack:{ctx.agent_id.key}"

    await rt.register("chat", chat_agent)

    # Simulate 3 waves of users — each wave is a different cohort
    wave_sizes = [100, 150, 250]
    offset     = 0

    for wave_num, wave_size in enumerate(wave_sizes, start=1):
        t0 = time.perf_counter()
        await asyncio.gather(*[
            rt.send_message("hi", recipient=AgentId("chat", f"s-{offset+i}"))
            for i in range(wave_size)
        ])
        elapsed = (time.perf_counter() - t0) * 1000
        offset += wave_size
        print(f"  Wave {wave_num}: {wave_size:3d} sessions in {elapsed:5.0f}ms"
              f"  | cumulative actors: {len(rt._agents_started)}")

    total = sum(wave_sizes)
    print(f"\n  Total sessions served       : {total}")
    print(f"  Actors in runtime           : {len(rt._agents_started)}")
    print(f"  dispatcher registered agents: {len(rt._dispatcher.registered_agents)}")

    # stop() tears down everything — dispatcher and mailboxes are cleared
    await rt.stop()

    print(f"\n  After stop():")
    print(f"    _agents_started : {rt._agents_started}  (empty)")
    print(f"    _started        : {rt._started}")
    print(f"  All {total} sessions garbage collected with one call")

await demo_session_lifecycle()

## 8. Head-to-Head: Why LangGraph Fails for Each Scenario

A direct comparison table reproduced as code that you can run and verify.

In [ ]:
# This cell doesn't run benchmarks — it prints the architectural comparison

comparisons = [
    {
        "scenario": "Session isolation",
        "langgraph": "Separate CompiledGraph instance per user. You manage the dict.",
        "actor":     "AgentId(type, key=session_id). Framework routes automatically.",
        "crewai":    "No session concept. All agents share process-level state.",
    },
    {
        "scenario": "500 concurrent users",
        "langgraph": "invoke() is synchronous by default. astream() doesn't parallelize sessions.",
        "actor":     "asyncio.gather() over 500 AgentIds — runs in parallel, ~50ms total.",
        "crewai":    "Sequential task execution. Crew.kickoff() blocks until all tasks done.",
    },
    {
        "scenario": "One session crashes",
        "langgraph": "Exception propagates out of graph.invoke(). Other sessions unaffected only if you catch it.",
        "actor":     "HandlerError isolated to that actor. Other sessions' mailboxes untouched.",
        "crewai":    "Exception propagates to Crew level. Whole crew may halt.",
    },
    {
        "scenario": "Per-user token streaming",
        "langgraph": "astream() streams between graph nodes. To stream tokens inside a node, you need callbacks/queues you wire by hand.",
        "actor":     "StreamPublisher(rt, TopicId('tokens', session_id)). Each user subscribes to their own topic.",
        "crewai":    "No native streaming. Requires custom callback hooks.",
    },
    {
        "scenario": "Slow LLM / backpressure",
        "langgraph": "No mailbox. All invoke() calls fire immediately. You catch RateLimitError inside nodes.",
        "actor":     "Mailbox(capacity=N). MailboxFullError on overflow — fail fast at the boundary.",
        "crewai":    "No backpressure concept. Tasks queue in Python lists, unbounded.",
    },
    {
        "scenario": "Session cleanup",
        "langgraph": "Checkpointer keeps state forever unless you call delete_thread() per thread.",
        "actor":     "rt.stop() tears down all actors and mailboxes in one call.",
        "crewai":    "No session lifecycle — objects stay alive until Python GC.",
    },
]

print("=" * 90)
for c in comparisons:
    print(f"\nScenario : {c['scenario']}")
    print(f"  LangGraph : {c['langgraph']}")
    print(f"  CrewAI    : {c['crewai']}")
    print(f"  Actor     : {c['actor']}  ← wins")
print("=" * 90)

## Summary

| Metric | LangGraph | CrewAI | Actor Runtime |
|---|---|---|---|
| Session isolation primitive | Manual dict + thread_id | None | `AgentId.key` |
| 500 concurrent sessions | ~25s (sequential) | ~25s+ (sequential) | ~50ms (concurrent) |
| Crash blast radius | 1 session (if caught) | Whole crew | 1 session (structural) |
| Per-user token streaming | Manual callback wiring | Not supported | `TopicId("tokens", session_id)` |
| Backpressure | None | None | `Mailbox(capacity=N)` |
| Session cleanup | `delete_thread()` per thread | Python GC | `rt.stop()` |
| Handler registration cost | 1 graph per user | 1 Crew per task | Once, for all users |

**The structural insight**: LangGraph/CrewAI model a workflow as a *unit of work*. The actor model models an *entity that lives between requests*. For multi-tenant chat, you need entities — sessions that persist across turns, in isolation, at scale.

The `AgentId(type, key)` is not just a routing detail — it is the architectural answer to multi-tenancy.